In [3]:
import numpy as np
from scipy.spatial import distance_matrix
from gurobipy import *
from numpy import random
import time
import pandas as pd
from haversine import haversine

# 1. 청룡동

In [4]:
입지후보지 = pd.read_csv('입지후보지.csv')
입지후보지 = 입지후보지[입지후보지['행정동'] == '청룡동']

버스 = pd.read_csv('청룡동_버스정류장.csv')
버스_points = np.array([list(i) for i in zip(버스['Y좌표'], 버스['X좌표'])])

노인시설 = pd.read_csv('청룡동_노인시설.csv', index_col=0)
노인시설_points = np.array([list(i) for i in zip(노인시설['위도'], 노인시설['경도'])])

노인복지시설 = pd.read_csv('청룡동_노인복지시설.csv', index_col=0)
노인복지시설_points = np.array([list(i) for i in zip(노인복지시설['위도'], 노인복지시설['경도'])])

경로당 = pd.read_csv('청룡동_경로당.csv', index_col=0)
경로당_points = np.array([list(i) for i in zip(경로당['위도'], 경로당['경도'])])

In [5]:
print(입지후보지.shape)
print(버스.shape)
print(노인시설.shape)
print(노인복지시설.shape)
print(경로당.shape)

(26, 6)
(103, 5)
(6, 5)
(6, 7)
(23, 5)


In [6]:
X = list(노인시설['위도']) + list(노인복지시설['위도']) + list(경로당['위도']) + list(버스['Y좌표'])
Y = list(노인시설['경도']) + list(노인복지시설['경도']) + list(경로당['경도']) + list(버스['X좌표'])
points = np.array([list(i) for i in zip(X, Y)])
print(points.shape)
points[:2]

(138, 2)


array([[ 36.80013535, 127.1776807 ],
       [ 36.76749349, 127.1650992 ]])

In [7]:
전체w = points.shape[0]
버스w = 버스.shape[0]
노인시설w = 노인시설.shape[0]
노인복지시설w = 노인복지시설.shape[0]
경로당w = 경로당.shape[0]

# 가중치는 해당수요지점수 / 전체수요지점수

m1 = 버스w / 전체w
m2 = 노인시설w / 전체w
m3 = 노인복지시설w / 전체w
m4 = 경로당w / 전체w

In [8]:
# haversine -> meter 단위로 수정
def mclp(버스_points, 노인시설_points, 노인복지시설_points, 경로당_points, points, K, radius):

    print('  Number of points %g' % points.shape[0])
    print('  K %g' % K)
    print('  Radius %g' % radius)

    start = time.time()
    sites = np.array([list(i) for i in zip(입지후보지['위도'], 입지후보지['경도'])])
    J = sites.shape[0]                             
    
    # 수요지점 수
    A = 버스_points.shape[0]
    B = 노인시설_points.shape[0]
    C = 노인복지시설_points.shape[0]
    D = 경로당_points.shape[0]

# 입지후보지와 수요지점 간 거리 계산
    D1 = []
    for i in 버스_points:
        site = []
        for j in sites:
            site.append(haversine(i, j)*1000)
        D1.append(site)
    D1 = np.array(D1)
    
    D2 = []
    for i in 노인시설_points:
        site = []
        for j in sites:
            site.append(haversine(i, j)*1000)
        D2.append(site)
    D2 = np.array(D2)    
    
    D3 = []
    for i in 노인복지시설_points:
        site = []
        for j in sites:
            site.append(haversine(i, j)*1000)
        D3.append(site)
    D3 = np.array(D3)
    
    D4 = []
    for i in 경로당_points:
        site = []
        for j in sites:
            site.append(haversine(i, j)*1000)
        D4.append(site)
    D4 = np.array(D4)

    for i in [D1, D2, D3, D4]:
        mask1 = i<=radius
        i[mask1]=1                                                 
        i[~mask1]=0

    m = Model()
    x1, x2, x3, x4 = {}, {}, {}, {}
    y = {}
    
    # 수요지점 변수 추가
    for i in range(A):                                       
        x1[i] = m.addVar(vtype=GRB.BINARY, name="x1%d" % i)
    for i in range(B):                                       
        x2[i] = m.addVar(vtype=GRB.BINARY, name="x2%d" % i)
    for i in range(C):                                       
        x3[i] = m.addVar(vtype=GRB.BINARY, name="x3%d" % i)
    for i in range(D):                                       
        x4[i] = m.addVar(vtype=GRB.BINARY, name="x4%d" % i)

    m = Model()
    x1, x2, x3, x4 = {}, {}, {}, {}
    y = {}
    
    # 수요지점 변수 추가
    for i in range(A):                                       
        x1[i] = m.addVar(vtype=GRB.BINARY, name="x1%d" % i)
    for i in range(B):                                       
        x2[i] = m.addVar(vtype=GRB.BINARY, name="x2%d" % i)
    for i in range(C):                                       
        x3[i] = m.addVar(vtype=GRB.BINARY, name="x3%d" % i)
    for i in range(D):                                       
        x4[i] = m.addVar(vtype=GRB.BINARY, name="x4%d" % i)

    
    # 입지후보지 변수 추가
    for j in range(J):
        y[j] = m.addVar(vtype=GRB.BINARY, name="y%d" % j)

    m.update()
    m.addConstr(quicksum(y[j] for j in range(J)) == K) 

    # # 수요지점 제약 조건
    for i in range(A): 
        m.addConstr(quicksum(y[j] for j in np.where(D1[i]==1)[0]) >= x1[i])
    for i in range(B): 
        m.addConstr(quicksum(y[j] for j in np.where(D2[i]==1)[0]) >= x2[i])
    for i in range(C): 
        m.addConstr(quicksum(y[j] for j in np.where(D3[i]==1)[0]) >= x3[i])
    for i in range(D): 
        m.addConstr(quicksum(y[j] for j in np.where(D4[i]==1)[0]) >= x4[i])
    
    # 목적함수 수정
    m.setObjective(quicksum(i for i in [m1*x1[a] for a in range(A)] + [m2*x2[b] for b in range(B)] + \
                            [m3*x3[c] for c in range(C)] + [m4*x4[d] for d in range(D)]), GRB.MAXIMIZE)
  

    m.setParam('OutputFlag', 0)
    m.optimize()
    end = time.time()
    print('----- Output -----')
    print('  Running time : %s seconds' % float(end-start))
    print('Optimal coverage points: %g' % m.objVal)

    solution = []
    if m.status == GRB.Status.OPTIMAL:
        for v in m.getVars():
            if v.x==1 and v.varName[0]=="y":
                solution.append(int(v.varName[1:]))
    opt_sites = sites[solution]
    return opt_sites,m.objVal

In [9]:
opts_sites, mobjVal = mclp(버스_points, 노인시설_points, 노인복지시설_points, 경로당_points, points, 5, 500)
opts_sites

  Number of points 138
  K 5
  Radius 500
Restricted license - for non-production use only - expires 2026-11-23
----- Output -----
  Running time : 0.969759464263916 seconds
Optimal coverage points: 47.7029


array([[ 36.78719082, 127.1499006 ],
       [ 36.7796    , 127.1459    ],
       [ 36.79660209, 127.160499  ],
       [ 36.78902469, 127.1570837 ],
       [ 36.78243638, 127.1622453 ]])

In [10]:
후보지 = pd.DataFrame(opts_sites, columns=['위도','경도'])
최종후보지 = pd.merge(입지후보지[['이름', '위도', '경도', '주소']], 후보지, on=['위도', '경도'], how='inner')
최종후보지.to_csv('청룡동_후보지(MCLP).csv', index=False, encoding='utf-8-sig')

# 2. 성환읍

In [11]:
입지후보지 = pd.read_csv('입지후보지.csv')
입지후보지 = 입지후보지[입지후보지['행정동'] == '성환읍']

버스 = pd.read_csv('성환읍_버스정류장.csv')
버스_points = np.array([list(i) for i in zip(버스['Y좌표'], 버스['X좌표'])])

노인시설 = pd.read_csv('성환읍_노인시설.csv', index_col=0)
노인시설_points = np.array([list(i) for i in zip(노인시설['위도'], 노인시설['경도'])])

노인복지시설 = pd.read_csv('성환읍_노인복지시설.csv', index_col=0)
노인복지시설_points = np.array([list(i) for i in zip(노인복지시설['위도'], 노인복지시설['경도'])])

경로당 = pd.read_csv('성환읍_경로당.csv', index_col=0)
경로당_points = np.array([list(i) for i in zip(경로당['위도'], 경로당['경도'])])

In [12]:
print(입지후보지.shape)
print(버스.shape)
print(노인시설.shape)
print(노인복지시설.shape)
print(경로당.shape)

(47, 6)
(165, 5)
(11, 5)
(9, 7)
(71, 5)


In [13]:
X = list(노인시설['위도']) + list(노인복지시설['위도']) + list(경로당['위도']) + list(버스['Y좌표'])
Y = list(노인시설['경도']) + list(노인복지시설['경도']) + list(경로당['경도']) + list(버스['X좌표'])
points = np.array([list(i) for i in zip(X, Y)])
print(points.shape)
points[:2]

(256, 2)


array([[ 36.94717367, 127.1513989 ],
       [ 36.92117221, 127.1374003 ]])

In [14]:
전체w = points.shape[0]
버스w = 버스.shape[0]
노인시설w = 노인시설.shape[0]
노인복지시설w = 노인복지시설.shape[0]
경로당w = 경로당.shape[0]

# 가중치는 해당수요지점수 / 전체수요지점수

m1 = 버스w / 전체w
m2 = 노인시설w / 전체w
m3 = 노인복지시설w / 전체w
m4 = 경로당w / 전체w

In [15]:
opts_sites, mobjVal = mclp(버스_points, 노인시설_points, 노인복지시설_points, 경로당_points, points, 5, 500)
opts_sites

  Number of points 256
  K 5
  Radius 500
----- Output -----
  Running time : 0.03812813758850098 seconds
Optimal coverage points: 38.8281


array([[ 36.93977962, 127.1529306 ],
       [ 36.92297813, 127.1196234 ],
       [ 36.91960656, 127.1328865 ],
       [ 36.90274734, 127.1093742 ],
       [ 36.91291818, 127.1327308 ]])

In [16]:
후보지 = pd.DataFrame(opts_sites, columns=['위도','경도'])
최종후보지 = pd.merge(입지후보지[['이름', '위도', '경도', '주소']], 후보지, on=['위도', '경도'], how='inner')
최종후보지.to_csv('성환읍_후보지(MCLP).csv', index=False, encoding='utf-8-sig')

# 3. 신안동

In [17]:
입지후보지 = pd.read_csv('입지후보지.csv')
입지후보지 = 입지후보지[입지후보지['행정동'] == '신안동']

버스 = pd.read_csv('신안동_버스정류장.csv')
버스_points = np.array([list(i) for i in zip(버스['Y좌표'], 버스['X좌표'])])

노인시설 = pd.read_csv('신안동_노인시설.csv', index_col=0)
노인시설_points = np.array([list(i) for i in zip(노인시설['위도'], 노인시설['경도'])])

노인복지시설 = pd.read_csv('신안동_노인복지시설.csv', index_col=0)
노인복지시설_points = np.array([list(i) for i in zip(노인복지시설['위도'], 노인복지시설['경도'])])

경로당 = pd.read_csv('신안동_경로당.csv', index_col=0)
경로당_points = np.array([list(i) for i in zip(경로당['위도'], 경로당['경도'])])

In [18]:
print(입지후보지.shape)
print(버스.shape)
print(노인시설.shape)
print(노인복지시설.shape)
print(경로당.shape)

(24, 6)
(73, 5)
(6, 5)
(1, 7)
(18, 5)


In [19]:
X = list(노인시설['위도']) + list(노인복지시설['위도']) + list(경로당['위도']) + list(버스['Y좌표'])
Y = list(노인시설['경도']) + list(노인복지시설['경도']) + list(경로당['경도']) + list(버스['X좌표'])
points = np.array([list(i) for i in zip(X, Y)])
print(points.shape)
points[:2]

(98, 2)


array([[ 36.81772127, 127.1525376 ],
       [ 36.81753212, 127.1518383 ]])

In [20]:
전체w = points.shape[0]
버스w = 버스.shape[0]
노인시설w = 노인시설.shape[0]
노인복지시설w = 노인복지시설.shape[0]
경로당w = 경로당.shape[0]

# 가중치는 해당수요지점수 / 전체수요지점수

m1 = 버스w / 전체w
m2 = 노인시설w / 전체w
m3 = 노인복지시설w / 전체w
m4 = 경로당w / 전체w

In [21]:
opts_sites, mobjVal = mclp(버스_points, 노인시설_points, 노인복지시설_points, 경로당_points, points, 5, 500)
opts_sites

  Number of points 98
  K 5
  Radius 500
----- Output -----
  Running time : 0.011504411697387695 seconds
Optimal coverage points: 41.6735


array([[ 36.81806566, 127.1561565 ],
       [ 36.82908494, 127.1675522 ],
       [ 36.82367503, 127.1554253 ],
       [ 36.82827909, 127.1856879 ],
       [ 36.84020122, 127.1793179 ]])

In [22]:
후보지 = pd.DataFrame(opts_sites, columns=['위도','경도'])
최종후보지 = pd.merge(입지후보지[['이름', '위도', '경도', '주소']], 후보지, on=['위도', '경도'], how='inner')
최종후보지.to_csv('신안동_후보지(MCLP).csv', index=False, encoding='utf-8-sig')